In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ===========================================================================
# STEP 2: Install Llamatelemetry
# ===========================================================================

!pip install -q --no-cache-dir --force-reinstall git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

import llamatelemetry
print("\nllamatelemetry version:", llamatelemetry.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 288.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 230.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 297.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 306.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 140.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 275.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 303.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 313.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/20


🎯 llamatelemetry v0.1.1 First-Time Setup - Kaggle 2× T4 Multi-GPU

🎮 GPU Detected: Tesla T4 (Compute 7.5)
  ✅ Tesla T4 detected - Perfect for llamatelemetry v0.1.1!
🌐 Platform: Kaggle

📦 Downloading Kaggle 2× T4 binaries (~961 MB)...
    Features: FlashAttention + Tensor Cores + Multi-GPU tensor-split

➡️  Attempt 1: HuggingFace (llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz)
📥 Downloading v0.1.1 from HuggingFace Hub...
   Repo: waqasm86/llamatelemetry-binaries
   File: v0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/1.40G [00:00<?, ?B/s]

v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/130 [00:00<?, ?B/s]

🔐 Verifying SHA256 checksum...
   ✅ Checksum verified
📦 Extracting llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz...
Found 21 files in archive
Extracted 21 files to /root/.cache/llamatelemetry/extract_0.1.1
✅ Extraction complete!
  Found bin/ and lib/ under /root/.cache/llamatelemetry/extract_0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2
  Copied 13 binaries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12
  Copied 2 libraries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/lib
✅ Binaries installed successfully!


llamatelemetry version: 0.1.1


In [2]:
import llamatelemetry as lt

# 1. Check GPU
cuda_info = lt.detect_cuda()
assert cuda_info["available"], "No CUDA GPU found"
print(f"GPU: {cuda_info['gpus'][0]['name']}")

GPU: Tesla T4


In [10]:
engine = lt.InferenceEngine(
    server_url="http://127.0.0.1:8080",  # default server address
    enable_telemetry=True,               
)

In [11]:
engine.unload_model()

engine.load_model(
    "gemma-3-1b-Q4_K_M",   # registry name
    auto_start=True,         # start llama-server after loading
    auto_configure=True,     # auto-set gpu_layers, ctx_size based on GPU
    gpu_layers=99,         # None = auto-detect from VRAM
    ctx_size=4096,           # None = auto-detect
    n_parallel=4,           # number of parallel request slots
    verbose=True,            # print loading progress
    report_suitability=True,
    embedding=True,  # IMPORTANT: enables /v1/embeddings
    pooling="mean"  # this fixes /v1/embeddings
)


Loading model: gemma-3-1b-Q4_K_M
✓ Using cached model: gemma-3-1b-it-Q4_K_M.gguf

GGUF Suitability Report:
  Model: Gemma-3-1B-It
  Size: 0.751 GB
  Quant: unknown
  Fits: True
  Recommended Quant: Q8_0
  Recommended GPU Layers: 99

Starting llama-server...
GPU Check:
  Platform: kaggle
  GPU: Tesla T4
  Compute Capability: 7.5
  Status: ✓ Compatible
  Command: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server -m /usr/local/lib/python3.12/dist-packages/llamatelemetry/models/gemma-3-1b-it-Q4_K_M.gguf --host 127.0.0.1 --port 8080 -ngl 99 -c 4096 --parallel 4 -b 512 -ub 128 --metrics --props --embeddings --pooling mean
Starting llama-server...
  Executable: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server
  Model: gemma-3-1b-it-Q4_K_M.gguf
  GPU Layers: 99
  Context Size: 4096
  Server URL: http://127.0.0.1:8080
Waiting for server to be ready...... ✓ Ready in 3.0s

✓ Model loaded and ready for inference
  Server: http://

True

In [12]:
from llamatelemetry import InferenceEngine
from llamatelemetry.embeddings import EmbeddingEngine

embedder = EmbeddingEngine(engine)

embedding = embedder.embed("AI is transforming the world")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1152
First 5 values: [ 0.00517522  0.00697909  0.01545575 -0.01180242  0.02807245]


In [13]:
from llamatelemetry.api import LlamaCppClient
client = LlamaCppClient(base_url="http://127.0.0.1:8080")

In [15]:
resp = client._request(
  "POST",
  "/embedding",
  json_data={"content": "Hello world", "pooling": "mean"}
)

# Handle the nested structure
if isinstance(resp, dict) and "embedding" in resp:
  # If it's a dict with embedding key
  embedding = resp["embedding"]
elif isinstance(resp, list) and len(resp) > 0 and isinstance(resp[0], dict) and "embedding" in resp[0]:
  # Handle the case where it's a list of dicts with embedding
  embedding = resp[0]["embedding"][0]  # Extract the actual embedding array
else:
  embedding = resp  # server returned raw list

print("Embedding dimension:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding dimension: 1152
First 5 values: [0.030108829960227013, -0.019661463797092438, -0.01509280689060688, 0.002736475085839629, -0.004213277716189623]


In [16]:
texts = ["Hello world", "Goodbye world"]
vectors = []

for t in texts:
  resp = client._request(
      "POST",
      "/embedding",
      json_data={"content": t, "pooling": "mean"}
  )
  if isinstance(resp, dict) and "embedding" in resp:
      vectors.append(resp["embedding"])
  else:
      vectors.append(resp)

print("Vectors:", len(vectors))
print("Vector size:", len(vectors[0]))


Vectors: 2
Vector size: 1


In [17]:
embeddings = client.embeddings.create(
  input=["Hello world", "Goodbye world"],
  model="local-gguf"
)
print(len(embeddings.data[0].embedding))


1152


In [19]:
import requests
import json

r = requests.post(
  "http://127.0.0.1:8080/v1/embeddings",
  json={"input": ["Hello world"]}
)

print("Status code:", r.status_code)

# Parse the JSON response
data = r.json()

# Extract the embedding (assuming it's in the first result)
if "data" in data and len(data["data"]) > 0:
    embedding = data["data"][0]["embedding"]
    print("Embedding dimension:", len(embedding))
    print("First 5 values:", embedding[:5])
    
    # Optionally print truncated usage info
    if "usage" in data:
        print("Usage:", data["usage"])
else:
    print("Full response:", json.dumps(data, indent=2)[:500] + "...")

Status code: 200
Embedding dimension: 1152
First 5 values: [0.030108829960227013, -0.019661463797092438, -0.01509280689060688, 0.002736475085839629, -0.004213277716189623]
Usage: {'prompt_tokens': 3, 'total_tokens': 3}


In [20]:
from llamatelemetry.embeddings import EmbeddingEngine

embedder = EmbeddingEngine(engine)
embeddings = embedder.embed_batch(["Hello world", "Goodbye world"])

print("Vectors:", embeddings.shape)
print("Vector size:", embeddings.shape[1])

Vectors: (2, 1152)
Vector size: 1152


In [ ]:
import requests

r = requests.post(
  "http://127.0.0.1:8080/v1/embeddings",
  json={"input": ["Hello world", "Goodbye world"]}
)
print(r.status_code, r.text)


In [22]:
def extract_embedding(resp):
  # Case 1: OpenAI-style dict
  if isinstance(resp, dict):
      if "data" in resp and resp["data"]:
          item = resp["data"][0]
          if isinstance(item, dict) and "embedding" in item:
              return item["embedding"]
      # Native dict
      if "embedding" in resp:
          return resp["embedding"]

  # Case 2: list response
  if isinstance(resp, list):
      # list of dicts
      if resp and isinstance(resp[0], dict) and "embedding" in resp[0]:
          return resp[0]["embedding"]
      # raw list of floats
      return resp

  raise ValueError(f"Unknown embedding response shape: {type(resp)}")


In [ ]:
resp = client._request(
  "POST",
  "/embedding",
  json_data={"content": "Hello world", "pooling": "mean"}
)

embedding = extract_embedding(resp)
print("Embedding dimension:", len(embedding))
print("First 5 values:", embedding[:5])


In [24]:
embeddings = client.embeddings.create(
  input=["Hello world", "Goodbye world"],
  model="local-gguf"
)

embedding = embeddings.data[0].embedding
print("Embedding dimension:", len(embedding))
print("First 5 values:", embedding[:5])


Embedding dimension: 1152
First 5 values: [0.030061015859246254, -0.019050918519496918, -0.00695616751909256, 0.0009638793999329209, -0.0066597978584468365]


In [25]:
texts = ["Hello world", "Goodbye world"]
vectors = []

for t in texts:
  resp = client._request(
      "POST",
      "/embedding",
      json_data={"content": t, "pooling": "mean"}
  )
  vectors.append(extract_embedding(resp))

print("Vectors:", len(vectors))
print("Vector size:", len(vectors[0]))


Vectors: 2
Vector size: 1


In [26]:
tokens = client.tokenize("Hello, llamatelemetry!")

print(f"Token count: {len(tokens.tokens)}")
print(f"Token IDs: {tokens.tokens}")

text = client.detokenize(tokens.tokens)
print(f"Detokenized: {text}")

Token count: 6
Token IDs: [9259, 236764, 20871, 79737, 54539, 236888]
Detokenized: Hello, llamatelemetry!


In [27]:
tokens = client.tokenize("Hello, llamatelemetry!", with_pieces=True)
print(tokens.tokens)

[{'id': 9259, 'piece': 'Hello'}, {'id': 236764, 'piece': ','}, {'id': 20871, 'piece': ' llam'}, {'id': 79737, 'piece': 'atele'}, {'id': 54539, 'piece': 'metry'}, {'id': 236888, 'piece': '!'}]


In [28]:
resp = client.chat.completions.create(
  messages=[{"role": "user", "content": "Explain flash attention step by step."}],
  max_tokens=256,
)

print(resp.choices[0].message.content)


Okay, let's break down Flash Attention, a groundbreaking technique designed to dramatically speed up the training of large language models (LLMs). It’s been a huge topic in recent research and has shown impressive results. Here's a breakdown of how it works, broken down into steps:

**1. The Problem: Traditional Attention Bottleneck**

Before Flash Attention, attention mechanisms were the backbone of LLMs, allowing them to focus on relevant parts of the input sequence when generating text. However, they have a significant bottleneck: **the quadratic complexity of self-attention**.  This means the computational cost of attending to every element in the input sequence grows quadratically with the length of the sequence. For long sequences (like entire documents or large language models), this becomes prohibitively slow, limiting the size and capabilities of LLMs.

**2. Flash Attention's Core Idea: Parallelizable and Memory-Efficient**

Flash Attention tackles this problem by introducing 